In [1]:
import numpy as np

def two_combinations(vector):
    """
    Convert an n-value vector to an n^2 2-dimensional vector representing all 2-combinations.

    :param vector: An n-value vector (list or numpy array).
    :return: An n^2 2-dimensional vector (2D numpy array).
    """
    n = len(vector)
    combination_vectors = []

    for i in range(n):
        for j in range(n):
            combination_vectors.append([vector[i], vector[j]])

    return np.array(combination_vectors)


In [48]:
import numpy as np
from scipy.stats import expon

def simulate_markov_batch(Q, initial_states, times, num_simulations):
    num_states = len(Q)
    num_times = len(times)
    states_at_times = np.zeros((num_simulations, num_times), dtype=int)

    for sim in range(num_simulations):
        state = initial_states
        states_at_times[sim, 0] = state

        for i in range(1, num_times):
            current_time = times[i - 1]
            end_time = times[i]
            while current_time < end_time:
                rate = -Q[state, state]
                time_to_next = expon.rvs(scale=1/rate)

                current_time += time_to_next
                if current_time < end_time:
                    transition_probs = Q[state, :] / rate
                    transition_probs[state] = 0
                    state = np.random.choice(num_states, p=transition_probs)

            states_at_times[sim, i] = state

    return states_at_times




/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


2

In [69]:
import torch
import torch.nn.functional as F
from torch import nn, Tensor
import numpy as np
from torch.nn import Module, Linear, BatchNorm1d, Tanh
from numba import cuda
from torch import autograd
import torch.nn as nn
import torch.nn.functional as F
import math
"""
In this section we have used code available online in : https://github.com/frankhan91/DeepBSDE

"""

class StochasticProcess:
    """
    Base class for defining PDE related function.

    Args:
    eqn_config (dict): dictionary containing PDE configuration parameters

    Attributes:
    dim (int): dimensionality of the problem
    total_time (float): total time horizon
    num_time_interval (int): number of time steps
    delta_t (float): time step size
    sqrt_delta_t (float): square root of time step size
    y_init (None): initial value of the function
    """

    def __init__(self, eqn_config: dict):
        self.dim = eqn_config['dim']
        self.total_time = eqn_config['total_time']
        self.num_time_interval = eqn_config['num_time_interval']
        self.delta_t = self.total_time / self.num_time_interval
        self.sqrt_delta_t = np.sqrt(self.delta_t)
        self.y_init = None

    def sample(self, num_sample: int) -> Tensor:
        """
        Sample forward SDE.

        Args:
        num_sample (int): number of samples to generate

        Returns:
        Tensor: tensor of size [num_sample, dim+1] containing samples
        """
        raise NotImplementedError

    def r_u(self, t: float, x: Tensor, y: Tensor, z: Tensor) -> Tensor:
        """
        Interest rate in the PDE.

        Args:
        t (float): current time
        x (Tensor): tensor of size [batch_size, dim] containing space coordinates
        y (Tensor): tensor of size [batch_size, 1] containing function values
        z (Tensor): tensor of size [batch_size, dim] containing gradients

        Returns:
        Tensor: tensor of size [batch_size, 1] containing generator values
        """
        raise NotImplementedError

    def h_z(self, t,x,y,z: Tensor) -> Tensor:
        """
        Function to compute H(z) in the PDE.

        Args:
        h (float): value of H function
        z (Tensor): tensor of size [batch_size, dim] containing gradients

        Returns:
        Tensor: tensor of size [batch_size, dim] containing H(z)
        """
        raise NotImplementedError

    def terminal(self, x: Tensor) -> Tensor:
        """
        Terminal condition of the PDE.

        Args:
        t (float): current time
        x (Tensor): tensor of size [batch_size, dim] containing space coordinates

        Returns:
        Tensor: tensor of size [batch_size, 1] containing terminal values
        """
        raise NotImplementedError


class RFQ(StochasticProcess):
  """
  Args:
  eqn_config (dict): dictionary containing PDE configuration parameters
  """
  def __init__(self, basic_config,specific_config):
    super(RFQ, self).__init__(basic_config)

    self.n_liqiudity_state  = specific_config['nls']
    self.x_init = np.ones(self.dim) * specific_config['init']
    self.lamdas = specific_config['lamdas']
    self.lamda_initial_state = specific_config['init_state'] #integer
    self.sigma = specific_config['sigma']
    self.k = specific_config['k']
    self.Q = specific_config['Q']

  def sample(self, num_sample)-> tuple:
    """
    Sample forward SDE.

    Args:
    num_sample (int): number of samples to generate

    Returns:
    tuple: tuple of two tensors: dw_sample of size [num_sample, dim, num_time_interval] and
    x_sample of size [num_sample, dim, num_time_interval+1]
    """

    dw_sample = np.random.normal(size=[num_sample,  self.num_time_interval]) * self.sqrt_delta_t
    x_sample = np.zeros([num_sample, self.num_time_interval + 1])
    x_sample[:, 0] = np.ones(num_sample) * self.x_init

    select_Q = np.ones([num_sample, self.n_liqiudity_state**2  , self.n_liqiudity_state**2  ]) * np.expand_dims(np.exp(self.Q * self.delta_t), axis=0)
    select_lamda = np.ones([num_sample, self.n_liqiudity_state**2 , 2]) * np.expand_dims(two_combinations(self.lamdas), axis=0)
    lamda_process = simulate_markov_batch(self.Q,self.lamda_initial_state,np.array([i*self.delta_t for i in range(self.num_time_interval)]) ,num_sample )
    ask_lamda = np.array([[self.lamdas[x // 2]for x in y] for y in  lamda_process ])
    bid_lamda = np.array([[self.lamdas[x % 2]for x in y] for y in  lamda_process ])

    for i in range(self.num_time_interval):

        x_sample[:,  i + 1] =  x_sample[:,  i ]+ (ask_lamda[:, i] -  bid_lamda[:, i]) * self.k * self.delta_t + self.sigma * dw_sample[:, i]
    return lamda_process, x_sample

  def r_u(self, t, x, y, z)-> torch.Tensor:
    """
    Interest rate in the PDE.

    Args:
    t (float): current time
    x (torch.Tensor): tensor of size [batch_size, dim] containing space coordinates
    y (torch.Tensor): tensor of size [batch_size, 1] containing function values
    z (torch.Tensor): tensor of size [batch_size, dim] containing gradients

    Returns:
    torch.Tensor: tensor of size [batch_size, 1] containing generator values
    """

    return 0

  def h_z(self,t,x,y,z)-> torch.Tensor:
      """
      Function to compute $h^T Z$ in the PDE.

      Args:
      t (float): current time
      x (torch.Tensor): tensor of size [batch_size, dim] containing space coordinates
      y (torch.Tensor): tensor of size [batch_size, 1] containing function value
      z (torch.Tensor): tensor of size [batch_size, dim] containing gradients

      Returns:
      torch.Tensor: tensor of size [batch_size, 1] containing H(z)
      """
      return 0


  def terminal(self,  x)-> torch.Tensor:
    """
    Terminal condition of the PDE.

    Args:
    t (float): current time
    x (torch.Tensor): tensor of size [batch_size, dim] containing space coordinates

    Returns:
    torch.Tensor: tensor of size [batch_size, 1] containing terminal values
    """
    return 0



In [70]:
"""A trading environment"""
import gym
from gym import spaces
from gym.utils import seeding

import numpy as np




class BoudEnv(gym.Env):
    """
    trading environment;
    """

    # trade_freq in unit of day, e.g 2: every 2 day; 0.5 twice a day;
    def __init__(self,basic_config,specific_config, num_sim=100):

        # simulated data: array of asset price, option price and delta paths (num_path x num_period)
        # generate data now
        self.sp = RFQ(basic_config,specific_config)



    def sample(self, batch_size):
      return  self.sp.sample(batch_size)




In [43]:
def weights_init_uniform_rule(m):
        classname = m.__class__.__name__
        # for every Linear layer in a model..
        if classname.find('nn.Linear') != -1:
            # get the number of the inputs
            n = m.in_features
            y = 1.0/np.sqrt(n)
            m.weight.data.uniform_(-y, y)
            m.bias.data.fill_(0)

class RL_Net(nn.Module):
    def __init__(self,INPUT_DIM,OUTPUT_DIM,HIDDEN_DIM , First = False):
        super(RL_Net, self).__init__()
        self.input_dim = INPUT_DIM
        self.output_dim = OUTPUT_DIM
        self.hidden_dim = HIDDEN_DIM
        self.is_first = First
        current_dim = self.input_dim
        self.layers = nn.ModuleList()
        self.bn = nn.ModuleList()
        self.droupout = nn.ModuleList() #drop out layer for regularization
        for hdim in self.hidden_dim:
            self.layers.append(nn.Linear(int(current_dim), int(hdim)))
            self.bn.append(nn.BatchNorm1d(int(hdim)))
            self.droupout.append(nn.Dropout(0.25)) # add a dropout layer
            current_dim = hdim
        self.layers.append(nn.Linear(int(current_dim), int(self.output_dim)))
    def forward(self, x):
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            if self.is_first == False:
              x = self.bn[i](x)
            x = F.relu(x)
        out = self.layers[-1](x)
        return out

In [66]:
Q = np.array([[-14.01, 4.37,4.37,5.27] , [19.32, -60.91,12.54,29.05] , [19.32,12.54,-60.91,29.05] , [23.67,15.00,15.00,-53.67]])

In [71]:
basic = {'dim' : 1 , 'total_time' : 30 , 'num_time_interval': 30 }
specific = {'init':103.593 , 'sigma':0.1839/np.sqrt(252) ,  'nls' :2 , 'lamdas' : np.array([10.83, 73.03]) , 'init_state':2 ,'k' : 2.29 , 'Q': Q}

env = BoudEnv(basic , specific)

In [72]:
env.sample(10)

(array([[2, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 3, 0, 1,
         3, 3, 3, 3, 0, 0, 2, 0],
        [2, 3, 3, 0, 0, 0, 1, 0, 0, 3, 2, 3, 0, 0, 2, 0, 3, 3, 0, 0, 3, 0,
         2, 3, 0, 0, 3, 0, 2, 0],
        [2, 0, 2, 0, 0, 2, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0,
         0, 0, 3, 0, 0, 0, 2, 0],
        [2, 3, 3, 2, 1, 0, 0, 1, 0, 3, 2, 0, 0, 3, 0, 0, 0, 1, 0, 0, 2, 1,
         3, 3, 0, 0, 3, 2, 0, 0],
        [2, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 3, 0, 0, 0, 1, 1, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 3, 0],
        [2, 0, 0, 0, 1, 0, 0, 2, 1, 0, 0, 3, 0, 0, 1, 2, 1, 3, 0, 1, 1, 3,
         0, 0, 0, 3, 0, 0, 0, 1],
        [2, 0, 2, 0, 3, 0, 0, 2, 0, 2, 2, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 1,
         0, 1, 3, 1, 0, 0, 1, 2],
        [2, 0, 0, 0, 1, 0, 2, 0, 0, 0, 1, 3, 0, 1, 0, 3, 3, 0, 0, 2, 3, 0,
         0, 1, 3, 0, 2, 1, 0, 0],
        [2, 3, 0, 3, 0, 0, 2, 0, 0, 0, 0, 2, 0, 3, 3, 3, 0, 0, 0, 0, 0, 0,
         0, 3, 2, 0, 0, 0, 0, 0],
        [2, 3, 0, 0